In [ ]:
import requests
import os
import mysql.connector as mc
import pandas as pd
from datetime import datetime
import dotenv
import time
import calendar

dotenv.load_dotenv()


DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_DATABASE'),
    'autocommit': True,
}

def create_table():
    conn = mc.connect(**DB_CONFIG)
    cursor = conn.cursor()
    create_query = """
    CREATE TABLE IF NOT EXISTS speed_pattern_timezone (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,
        statDate VARCHAR(6) NOT NULL, 
        roadName VARCHAR(100),
        direction VARCHAR(20),
        sectionName VARCHAR(200),
        hour00 INT, hour01 INT, hour02 INT, hour03 INT, hour04 INT, hour05 INT,
        hour06 INT, hour07 INT, hour08 INT, hour09 INT, hour10 INT, hour11 INT,
        hour12 INT, hour13 INT, hour14 INT, hour15 INT, hour16 INT, hour17 INT,
        hour18 INT, hour19 INT, hour20 INT, hour21 INT, hour22 INT, hour23 INT,
        UNIQUE KEY uniq_timezone (statDate, roadName, direction, sectionName)
    );
    """
    try:
        cursor.execute(create_query)
        conn.commit()
        print("[INFO] speed_pattern_timezone 테이블 생성 완료")
    except Exception as e:
        print(f"[ERROR] 테이블 생성 실패: {e}")
    finally:
        cursor.close()
        conn.close()

create_table()

[INFO] speed_pattern_timezone 테이블 생성 완료


In [3]:

BASE_URL = 'http://apis.data.go.kr/6280000/ICRoadStat_v2/STAT-Speed_DD_Section'
SERVICE_KEY = os.getenv('PUBLIC_DATA_API_KEY')
MAX_PAGE_SIZE = 5000

ROAD_NAMES = [
    '경인로', '봉오대로', '아암대로', '인주대로', '중봉대로', 
    '경원대로', '남동대로', '무네미로', '비류대로', '드림로',
    '서해대로', '수봉로', '장제로', '백범로', '호구포로'
]


hour_cols = [f"hour{str(i).zfill(2)}" for i in range(24)]
columns = ["statDate", "roadName", "direction", "sectionName"] + hour_cols

col_str = ", ".join(columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT IGNORE INTO speed_pattern_timezone ({col_str}) VALUES ({placeholders})"

def get_daily_road_data(ymd, road_name):
    """특정 일자(YMD), 특정 도로의 데이터 호출 [cite: 29, 40]"""
    params = {
        'serviceKey': SERVICE_KEY,
        'pageNo': 1,
        'numOfRows': MAX_PAGE_SIZE,
        'YMD': ymd,
        'roadName': road_name,
        '_type': 'json'
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=15)
        return response.json()
    except:
        return None

def extract_items(json_data):
    """기존 요일별 파싱 로직을 유지하며 시간대 데이터를 추출합니다. [cite: 35, 46]"""
    try:
        if not json_data: return []
        body = json_data.get('response', {}).get('body', {})
        items_entry = body.get('items', [])
        
        
        if isinstance(items_entry, dict):
            items = items_entry.get('item', [])
        else:
            items = items_entry

        if not items: return []
        if isinstance(items, dict): items = [items]
            
        processed = []
        for item in items:
            if not isinstance(item, dict): continue
            row = {
                'roadName': item.get('roadName'),
                'direction': item.get('direction'),
                'sectionName': item.get('sectionName')
            }
        
            for h_col in hour_cols:
                val = item.get(h_col, 0)
                try:
                    row[h_col] = int(float(val)) if val else 0
                except:
                    row[h_col] = 0
            processed.append(row)
        return processed
    except Exception as e:
        print(f"\n[파싱 에러 상세] {e}")
        return []

def main():
    conn = mc.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    
    years = [2023, 2024, 2025, 2026]
    total_count = 0

    for year in years:
        for month in range(1, 13):
            if year == 2026 and month > 3: break
            
            target_ym = f"{year}{str(month).zfill(2)}"
            last_day = calendar.monthrange(year, month)[1]
            all_month_data = [] 

            print(f"\n[평균 작업 시작] {target_ym} 시간대별 수집 중...")
            
           
            for day in range(1, last_day + 1):
                ymd = f"{target_ym}{str(day).zfill(2)}"
                for road in ROAD_NAMES:
                    data = get_daily_road_data(ymd, road)
                    items = extract_items(data)
                    if items:
                        all_month_data.extend(items)
                print(f"  - {ymd} 데이터 확보 완료", end="\r")

          
            if all_month_data:
                df = pd.DataFrame(all_month_data)
                # 도로/방향/구간별로 그룹화하여 시간대별 평균 계산
                avg_df = df.groupby(['roadName', 'direction', 'sectionName'])[hour_cols].mean().round().astype(int).reset_index()
                avg_df['statDate'] = target_ym
                
                final_rows = [tuple(x) for x in avg_df[columns].values]
                
                if final_rows:
                    cursor.executemany(insert_query, final_rows)
                    total_count += len(final_rows)
                    print(f"  => {target_ym} 평균 데이터 {len(final_rows)}건 저장 성공")
                
                time.sleep(0.5)

    print(f"\n\n[최종 결과] 총 {total_count}건의 월별 시간대 평균 데이터가 저장되었습니다.")
    cursor.close()
    conn.close()

if __name__ == "__main__":
    main()


[평균 작업 시작] 202301 시간대별 수집 중...
  => 202301 평균 데이터 34건 저장 성공

[평균 작업 시작] 202302 시간대별 수집 중...
  => 202302 평균 데이터 268건 저장 성공

[평균 작업 시작] 202303 시간대별 수집 중...
  => 202303 평균 데이터 270건 저장 성공

[평균 작업 시작] 202304 시간대별 수집 중...
  => 202304 평균 데이터 270건 저장 성공

[평균 작업 시작] 202305 시간대별 수집 중...
  => 202305 평균 데이터 270건 저장 성공

[평균 작업 시작] 202306 시간대별 수집 중...
  => 202306 평균 데이터 270건 저장 성공

[평균 작업 시작] 202307 시간대별 수집 중...
  => 202307 평균 데이터 270건 저장 성공

[평균 작업 시작] 202308 시간대별 수집 중...
  => 202308 평균 데이터 270건 저장 성공

[평균 작업 시작] 202309 시간대별 수집 중...
  => 202309 평균 데이터 270건 저장 성공

[평균 작업 시작] 202310 시간대별 수집 중...
  => 202310 평균 데이터 270건 저장 성공

[평균 작업 시작] 202311 시간대별 수집 중...
  => 202311 평균 데이터 270건 저장 성공

[평균 작업 시작] 202312 시간대별 수집 중...
  => 202312 평균 데이터 270건 저장 성공

[평균 작업 시작] 202401 시간대별 수집 중...
  => 202401 평균 데이터 270건 저장 성공

[평균 작업 시작] 202402 시간대별 수집 중...
  => 202402 평균 데이터 270건 저장 성공

[평균 작업 시작] 202403 시간대별 수집 중...
  => 202403 평균 데이터 270건 저장 성공

[평균 작업 시작] 202404 시간대별 수집 중...
  => 202404 평균 데이터 270건 저장 성공

[평균 작업 시